In [1]:
#!/usr/bin/env python3
"""
Poster-only rerender script for 4 larger figures from RF_PCA notebook.

Updated version:
- larger legends across plotted figures
- PC1 loadings now show top 5 positive and top 5 negative drivers
- PC1 loadings plot includes a clearer legend

Note:
- This uploaded file does not currently contain a SHAP plotting function,
  so the requested "make the SHAP plot thinner" change cannot be applied here.
  You would need to edit the separate SHAP plotting script/block.
"""

import os
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


# ============================================================
# 1) CONFIGURATION
# ============================================================
os.chdir('/scratch/liuhon33/parallel/AGPMicrobiomeHostPredictions')
OTU_FILE_PATH = "./Data/Cleaned_data/AGP_Otu_Data.csv"
METADATA_FILE_PATH = "./Data/Cleaned_data/processed_metadata.csv"
TAXA_MAP_PATH = "./Data/Raw_Data/taxa_md5.xls"

METADATA_INDEX_COL = "sample_name"
OTU_INDEX_COL = 0

OUTER_CV_SPLITS = 10
INNER_CV_SPLITS = 5
RANDOM_STATE_CV = 42
ABUNDANCE_THRESHOLD = 0.0001

# Poster figure settings
N_PCS_TO_EXPLORE = 150
N_PCS_FOR_NESTED_CV = 10
TOP_K_LOADINGS = 10
PERFORMANCE_PLOT_MODE = "pc1_to_pc10"   # "pc1_to_pc10" or "top10_best"
LEGEND_FONT_SIZE = 20

# Output folder for poster-friendly plots
OUTPUT_DIR = Path("./hongrui_result/RF_on_PCA_Results/poster_large_plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = OUTPUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NESTED_CACHE_PATH = CACHE_DIR / f"nested_cv_results_{N_PCS_FOR_NESTED_CV}pcs.pkl"

PARAM_GRID = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_features": ["sqrt", "log2"],
    "rf__min_samples_leaf": [5, 10, 20],
    "rf__max_depth": [None, 10, 20],
}

# Same exclusions as notebook
EXCLUDE_METADATA_COLS = [
    "cat",
    "dog",
    "multivitamin",
    "other_supplement_frequency",
    "cosmetics_frequency",
    "fermented_plant_frequency",
    "homecooked_meals_frequency",
    "meat_eggs_frequency",
    "sugary_sweets_frequency",
    "vivid_dreams",
    "sugar_sweetened_drink_frequency",
    # "free_sugar_scaled_0_5",
    "artificial_sweeteners",
    # "one_liter_of_water_a_day_frequency",
    "olive_oil",
    "prepared_meals_frequency",
    "ready_to_eat_meals_frequency",
    # "probiotic_frequency",
    # "whole_eggs",
]


# ============================================================
# 2) POSTER STYLE
# ============================================================
def apply_poster_style() -> None:
    """Set larger fonts and cleaner rendering for poster figures."""
    sns.set_style("whitegrid")
    plt.rcParams.update({
        "figure.dpi": 180,
        "savefig.dpi": 300,
        "font.size": 16,
        "axes.titlesize": 24,
        "axes.titleweight": "bold",
        "axes.labelsize": 20,
        "axes.labelweight": "bold",
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": LEGEND_FONT_SIZE,
        "legend.title_fontsize": LEGEND_FONT_SIZE,
        "lines.linewidth": 2.5,
        "axes.linewidth": 1.2,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def save_both(fig: plt.Figure, stem: str) -> None:
    """Save both PDF and PNG for poster use."""
    pdf_path = OUTPUT_DIR / f"{stem}.pdf"
    png_path = OUTPUT_DIR / f"{stem}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    print(f"Saved: {pdf_path}")
    print(f"Saved: {png_path}")


# ============================================================
# 3) DATA PREP (same logic as notebook)
# ============================================================
def make_unique(names):
    counts = {}
    out = []
    for n in names:
        k = counts.get(n, 0) + 1
        counts[n] = k
        out.append(n if k == 1 else f"{n}__{k}")
    return out



def load_and_prepare_data():
    print("--- Loading data ---")
    metadata_df = pd.read_csv(METADATA_FILE_PATH, index_col=METADATA_INDEX_COL)
    otu_df = pd.read_csv(OTU_FILE_PATH, index_col=OTU_INDEX_COL)

    numeric_metadata_cols = metadata_df.select_dtypes(include=[np.number]).columns
    metadata_df[numeric_metadata_cols] = metadata_df[numeric_metadata_cols].replace(5, np.nan)

    common_samples = metadata_df.index.intersection(otu_df.index)
    metadata_df = metadata_df.loc[common_samples]
    otu_df = otu_df.loc[common_samples]
    print(f"Data aligned. Found {len(common_samples)} common samples.")
    print(f"Initial number of OTUs: {otu_df.shape[1]}")

    taxa_df = pd.read_csv(
        TAXA_MAP_PATH,
        sep="\t",
        header=0,
        index_col=0,
        engine="python",
    )
    taxa_df.index = taxa_df.index.astype(str).str.strip()
    taxa_df["Family"] = taxa_df["Family"].fillna("UnclassifiedFamily").astype(str).str.strip()
    taxa_df["Genus"] = taxa_df["Genus"].fillna("UnclassifiedGenus").astype(str).str.strip()
    taxa_df["Family_Genus"] = taxa_df["Family"] + "_" + taxa_df["Genus"]
    mapper = taxa_df["Genus"].to_dict()

    otu_md5 = otu_df.columns.astype(str).str.strip()
    md5_prefix_len = 8
    new_cols = [f"{mapper.get(m, 'Unmapped')}__{m[:md5_prefix_len]}" for m in otu_md5]
    otu_df = otu_df.copy()
    otu_df.columns = make_unique(new_cols)

    otu_rel_abund = otu_df.apply(lambda x: x / x.sum(), axis=1)
    mean_rel_abund = otu_rel_abund.mean(axis=0)
    otus_to_keep = mean_rel_abund[mean_rel_abund > ABUNDANCE_THRESHOLD].index
    otu_df_filtered = otu_df[otus_to_keep]
    print(f"Number of OTUs after filtering: {otu_df_filtered.shape[1]}")

    otu_df_log_transformed = np.log1p(otu_df_filtered)
    print("Applied log(x+1) transformation to OTU counts.")

    numeric_cols = metadata_df.select_dtypes(include=np.number).columns.tolist()
    excluded_present = [c for c in EXCLUDE_METADATA_COLS if c in numeric_cols]
    if excluded_present:
        print(f"Dropping {len(excluded_present)} excluded predictors.")
    numeric_cols = [c for c in numeric_cols if c not in EXCLUDE_METADATA_COLS]

    X_metadata = metadata_df[numeric_cols].copy()
    print(f"Using {X_metadata.shape[1]} numeric metadata features as predictors.")

    return metadata_df, otu_df_filtered, otu_df_log_transformed, X_metadata


# ============================================================
# 4) EXPLORATORY PCA FOR SCREE + LOADINGS
# ============================================================
def run_exploratory_pca(otu_df_log_transformed: pd.DataFrame, n_pcs_to_explore: int = 150):
    print(f"\n--- Performing exploratory PCA for first {n_pcs_to_explore} PCs ---")
    scaler_otu = StandardScaler()
    otu_scaled = scaler_otu.fit_transform(otu_df_log_transformed)

    pca_otu_full = PCA(n_components=None)
    pca_otu_full.fit(otu_scaled)

    all_otu_explained_variance = pca_otu_full.explained_variance_ratio_
    otu_explained_variance_explore = all_otu_explained_variance[:n_pcs_to_explore]
    pca_otu_loadings_explore = pca_otu_full.components_[:n_pcs_to_explore, :]

    return otu_explained_variance_explore, pca_otu_loadings_explore


# ============================================================
# 5) NESTED CV (same model logic, smaller scope for poster)
# ============================================================
def run_nested_cv_results(X_metadata: pd.DataFrame,
                          otu_df_log_transformed: pd.DataFrame,
                          n_pcs_to_predict: int = 10,
                          use_cache: bool = True):
    if use_cache and NESTED_CACHE_PATH.exists():
        print(f"\n--- Loading cached nested CV results from {NESTED_CACHE_PATH} ---")
        return joblib.load(NESTED_CACHE_PATH)

    print(f"\n--- Running {OUTER_CV_SPLITS}-fold nested CV for {n_pcs_to_predict} PCs ---")
    nested_cv_results = {}

    outer_cv = KFold(n_splits=OUTER_CV_SPLITS, shuffle=True, random_state=RANDOM_STATE_CV)
    inner_cv = KFold(n_splits=INNER_CV_SPLITS, shuffle=True, random_state=RANDOM_STATE_CV)

    rf_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestRegressor(random_state=RANDOM_STATE_CV)),
    ])

    max_needed_pcs = max(n_pcs_to_predict, N_PCS_TO_EXPLORE)

    for pc_idx in range(n_pcs_to_predict):
        pc_name = f"PC{pc_idx + 1}"
        print(f"Processing {pc_name} ...")
        t0 = time.time()

        outer_loop_scores = []
        y_true_all_folds = []
        y_pred_all_folds = []

        for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_metadata), start=1):
            X_outer_train = X_metadata.iloc[train_idx]
            X_outer_test = X_metadata.iloc[test_idx]

            Y_outer_train_base = otu_df_log_transformed.iloc[train_idx]
            Y_outer_test_base = otu_df_log_transformed.iloc[test_idx]

            otu_scaler_fold = StandardScaler()
            Y_train_scaled = otu_scaler_fold.fit_transform(Y_outer_train_base)
            Y_test_scaled = otu_scaler_fold.transform(Y_outer_test_base)

            pca_fold = PCA(n_components=max_needed_pcs)
            pca_fold.fit(Y_train_scaled)

            Y_train_pcs = pca_fold.transform(Y_train_scaled)[:, :n_pcs_to_predict]
            Y_test_pcs = pca_fold.transform(Y_test_scaled)[:, :n_pcs_to_predict]

            y_outer_train = Y_train_pcs[:, pc_idx]
            y_outer_test = Y_test_pcs[:, pc_idx]

            grid_search = GridSearchCV(
                estimator=rf_pipe,
                param_grid=PARAM_GRID,
                cv=inner_cv,
                scoring="r2",
                n_jobs=-1,
                verbose=0,
            )
            grid_search.fit(X_outer_train, y_outer_train)

            best_model = grid_search.best_estimator_
            y_pred = best_model.predict(X_outer_test)
            score = r2_score(y_outer_test, y_pred)

            outer_loop_scores.append(score)
            y_true_all_folds.append(y_outer_test)
            y_pred_all_folds.append(y_pred)

            print(f"  fold {fold}/{OUTER_CV_SPLITS}: R^2 = {score:.4f}")

        nested_cv_results[pc_name] = {
            "avg_r2": float(np.mean(outer_loop_scores)),
            "std_r2": float(np.std(outer_loop_scores, ddof=1)),
            "fold_r2": [float(s) for s in outer_loop_scores],
            "y_true": np.concatenate(y_true_all_folds),
            "y_pred": np.concatenate(y_pred_all_folds),
        }

        dt = time.time() - t0
        print(
            f"  {pc_name}: mean R^2 = {nested_cv_results[pc_name]['avg_r2']:.4f} "
            f"+/- {nested_cv_results[pc_name]['std_r2']:.4f} ({dt:.1f}s)"
        )

    joblib.dump(nested_cv_results, NESTED_CACHE_PATH)
    print(f"Cached nested CV results to: {NESTED_CACHE_PATH}")
    return nested_cv_results


# ============================================================
# 6) PLOTTING
# ============================================================
def plot_scree(otu_explained_variance_explore: np.ndarray):
    x = np.arange(1, len(otu_explained_variance_explore) + 1)
    y = otu_explained_variance_explore * 100

    fig, ax = plt.subplots(figsize=(8.5, 7))
    ax.bar(x, y, width=0.9, label="Explained variance")
    ax.set_title("Variance Explained by Microbiome PC")
    ax.set_xlabel("Principal component")
    ax.set_ylabel("Variance explained (%)")
    ax.set_xlim(0.5, len(x) + 0.5)

    tick_step = 10 if len(x) >= 100 else 5
    ax.set_xticks(np.arange(1, len(x) + 1, tick_step))
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)
    ax.legend(frameon=True, fontsize=LEGEND_FONT_SIZE)

    fig.tight_layout()
    save_both(fig, "poster_scree_plot_pc1_to_pc150")
    plt.close(fig)



def plot_performance_bar(nested_cv_results: dict):
    results_df = pd.DataFrame([
        {
            "PC": pc,
            "R2": res["avg_r2"],
            "Error": res["std_r2"],
        }
        for pc, res in nested_cv_results.items()
    ])

    if PERFORMANCE_PLOT_MODE == "top10_best":
        perf_df = results_df.sort_values("R2", ascending=False).head(10).reset_index(drop=True)
        title = "Prediction Accuracy (Nested CV $R^2$) for Top 10 Best-Performing Microbiome PCs"
    else:
        perf_df = results_df.copy()
        perf_df["PC_num"] = perf_df["PC"].str.replace("PC", "", regex=False).astype(int)
        perf_df = perf_df.sort_values("PC_num").drop(columns="PC_num").reset_index(drop=True)
        title = "Prediction Accuracy (Nested CV $R^2$) for Microbiome PC"

    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.bar(
        perf_df["PC"],
        perf_df["R2"],
        yerr=perf_df["Error"],
        capsize=6,
        edgecolor="black",
        alpha=0.85,
        label="Mean nested-CV $R^2$",
    )

    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Target principal component")
    ax.set_ylabel("Average nested-CV $R^2$ (+/- SD across folds)")
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)
    ax.legend(frameon=True, fontsize=LEGEND_FONT_SIZE)

    for bar, r2 in zip(bars, perf_df["R2"]):
        y_pos = bar.get_height() + (0.008 if r2 >= 0 else -0.02)
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y_pos,
            f"{r2:.3f}",
            ha="center",
            va="bottom" if r2 >= 0 else "top",
            fontsize=13,
            fontweight="bold",
        )

    fig.tight_layout()
    save_both(fig, "poster_prediction_accuracy_top10")
    plt.close(fig)



def plot_pc1_loadings(otu_df_filtered: pd.DataFrame,
                      otu_explained_variance_explore: np.ndarray,
                      pca_otu_loadings_explore: np.ndarray,
                      top_k: int = 5):
    loadings = pd.Series(
        pca_otu_loadings_explore[0, :],
        index=otu_df_filtered.columns,
        name="loading",
    )

    top = pd.concat([
        loadings.nlargest(top_k),
        loadings.nsmallest(top_k),
    ]).sort_values()

    bar_colors = ["tab:blue" if v < 0 else "tab:orange" for v in top.values]

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top.index, top.values, edgecolor="black", alpha=0.9, color=bar_colors)
    ax.axvline(0, color="black", linewidth=1.2)
    ax.set_xlabel("Loading value")
    ax.set_ylabel("OTU identifier")
    ax.set_title(
        f"OTU Loadings for Microbiome PC1"
    )
    ax.grid(True, axis="x", linestyle="--", alpha=0.4)
    ax.tick_params(axis="y", labelsize=13)

    legend_handles = [
        Patch(facecolor="tab:orange", edgecolor="black", label="Positive loadings"),
        Patch(facecolor="tab:blue", edgecolor="black", label="Negative loadings"),
    ]

    fig.tight_layout()
    save_both(fig, "poster_pc1_loadings_top5_pos_neg")
    plt.close(fig)



def plot_pc1_true_vs_pred(nested_cv_results: dict):
    if "PC1" not in nested_cv_results:
        raise ValueError("PC1 not found in nested CV results.")

    true_values = np.asarray(nested_cv_results["PC1"]["y_true"])
    pred_values = np.asarray(nested_cv_results["PC1"]["y_pred"])
    avg_r2 = float(nested_cv_results["PC1"]["avg_r2"])

    line_model = LinearRegression()
    line_model.fit(true_values.reshape(-1, 1), pred_values)

    line_x = np.array([true_values.min(), true_values.max()]).reshape(-1, 1)
    line_y = line_model.predict(line_x)
    line_r2 = line_model.score(true_values.reshape(-1, 1), pred_values)

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.scatter(true_values, pred_values, alpha=0.45, s=28, label="Out-of-fold predictions")
    ax.plot(
        line_x.flatten(),
        line_y,
        "--",
        linewidth=2.5,
        label=f"Fitted line ($R^2$ = {line_r2:.3f})",
    )

    ax.set_title(
        f"Best-Performing Model: Predict PC1 from Lifestyle Variables\n"
        f"(Nested CV $R^2$ = {avg_r2:.3f})"
    )
    ax.set_xlabel("True PC1 value (held-out test folds)")
    ax.set_ylabel("Predicted PC1 value")
    ax.legend(frameon=True, fontsize=LEGEND_FONT_SIZE)
    ax.grid(True, linestyle="--", alpha=0.6)

    fig.tight_layout()
    save_both(fig, "poster_pc1_true_vs_predicted")
    plt.close(fig)


# ============================================================
# 7) MAIN
# ============================================================
def main():
    apply_poster_style()

    metadata_df, otu_df_filtered, otu_df_log_transformed, X_metadata = load_and_prepare_data()

    otu_explained_variance_explore, pca_otu_loadings_explore = run_exploratory_pca(
        otu_df_log_transformed,
        n_pcs_to_explore=N_PCS_TO_EXPLORE,
    )

    nested_cv_results = run_nested_cv_results(
        X_metadata,
        otu_df_log_transformed,
        n_pcs_to_predict=N_PCS_FOR_NESTED_CV,
        use_cache=True,
    )

    plot_scree(otu_explained_variance_explore)
    plot_performance_bar(nested_cv_results)
    plot_pc1_loadings(
        otu_df_filtered,
        otu_explained_variance_explore,
        pca_otu_loadings_explore,
        top_k=TOP_K_LOADINGS,
    )
    plot_pc1_true_vs_pred(nested_cv_results)

    print("\nDone. Poster-friendly plots are in:")
    print(OUTPUT_DIR.resolve())


if __name__ == "__main__":
    main()


--- Loading data ---
Data aligned. Found 9559 common samples.
Initial number of OTUs: 4460
Number of OTUs after filtering: 819
Applied log(x+1) transformation to OTU counts.
Dropping 15 excluded predictors.
Using 29 numeric metadata features as predictors.

--- Performing exploratory PCA for first 150 PCs ---

--- Loading cached nested CV results from hongrui_result/RF_on_PCA_Results/poster_large_plots/cache/nested_cv_results_10pcs.pkl ---
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/poster_scree_plot_pc1_to_pc150.pdf
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/poster_scree_plot_pc1_to_pc150.png
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/poster_prediction_accuracy_top10.pdf
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/poster_prediction_accuracy_top10.png
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/poster_pc1_loadings_top5_pos_neg.pdf
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/poster_pc1_loadings_t

In [2]:
# ============================================================
# NEW BLOCK: generalized loading plot for any PC
# Put this after your existing plotting functions
# ============================================================

from pathlib import Path
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# optional: save into a dedicated subfolder
LOADINGS_OUT_DIR = OUTPUT_DIR / "all_pc_loadings"
LOADINGS_OUT_DIR.mkdir(parents=True, exist_ok=True)

def plot_single_pc_loadings(
    otu_df_filtered: pd.DataFrame,
    pca_otu_loadings_explore: np.ndarray,
    otu_explained_variance_explore: np.ndarray = None,
    pc_num: int = 1,
    top_k: int = 5,
    save_dir: Path = None,
    figsize=(9, 7),
):
    """
    Plot top positive and negative OTU loadings for one PC.

    Parameters
    ----------
    otu_df_filtered : pd.DataFrame
        OTU table after filtering; columns must match PCA input features.
    pca_otu_loadings_explore : np.ndarray
        PCA loadings array of shape [n_pcs, n_features].
    otu_explained_variance_explore : np.ndarray, optional
        Explained variance ratio for each PC.
    pc_num : int
        PC number starting from 1.
    top_k : int
        Number of top positive and negative loadings to display.
    save_dir : Path
        Folder to save figure.
    figsize : tuple
        Figure size.
    """

    pc_idx = pc_num - 1
    if pc_idx < 0 or pc_idx >= pca_otu_loadings_explore.shape[0]:
        raise ValueError(f"PC{pc_num} is not available in pca_otu_loadings_explore.")

    loadings = pd.Series(
        pca_otu_loadings_explore[pc_idx, :],
        index=otu_df_filtered.columns,
        name="loading"
    )

    top = pd.concat([
        loadings.nlargest(top_k),
        loadings.nsmallest(top_k)
    ]).sort_values()

    bar_colors = ["tab:blue" if v < 0 else "tab:orange" for v in top.values]

    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(top.index, top.values, color=bar_colors, edgecolor="black", alpha=0.9)
    ax.axvline(0, color="black", linewidth=1.2)
    ax.set_xlabel("Loading value", fontsize=13)
    ax.set_ylabel("OTU identifier", fontsize=13)

    if otu_explained_variance_explore is not None and pc_idx < len(otu_explained_variance_explore):
        var_pct = otu_explained_variance_explore[pc_idx] * 100
        title = f"OTU Loadings for Microbiome PC{pc_num} ({var_pct:.2f}% variance explained)"
    else:
        title = f"OTU Loadings for Microbiome PC{pc_num}"

    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.grid(True, axis="x", linestyle="--", alpha=0.4)
    ax.tick_params(axis="y", labelsize=11)
    ax.tick_params(axis="x", labelsize=11)

    legend_handles = [
        Patch(facecolor="tab:orange", edgecolor="black", label="Positive loadings"),
        Patch(facecolor="tab:blue", edgecolor="black", label="Negative loadings"),
    ]
    ax.legend(handles=legend_handles, fontsize=11, loc="best")

    fig.tight_layout()

    if save_dir is not None:
        png_path = Path(save_dir) / f"pc{pc_num}_loadings_top{top_k}_pos_neg.png"
        pdf_path = Path(save_dir) / f"pc{pc_num}_loadings_top{top_k}_pos_neg.pdf"
        fig.savefig(png_path, dpi=300, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"Saved: {png_path}")
        print(f"Saved: {pdf_path}")

    plt.show()
    plt.close(fig)

In [7]:
# ============================================================
# RUN THIS FIRST if you want the variables available in notebook
# ============================================================

apply_poster_style()

metadata_df, otu_df_filtered, otu_df_log_transformed, X_metadata = load_and_prepare_data()

otu_explained_variance_explore, pca_otu_loadings_explore = run_exploratory_pca(
    otu_df_log_transformed,
    n_pcs_to_explore=N_PCS_TO_EXPLORE,
)

--- Loading data ---
Data aligned. Found 9559 common samples.
Initial number of OTUs: 4460
Number of OTUs after filtering: 819
Applied log(x+1) transformation to OTU counts.
Dropping 15 excluded predictors.
Using 29 numeric metadata features as predictors.

--- Performing exploratory PCA for first 150 PCs ---


In [8]:
# ============================================================
# NEW BLOCK: loop through PC1 to PC10 and save each separately
# ============================================================

N_PCS_TO_PLOT = 10
TOP_K_LOADINGS_EACH = 10   # top 5 positive + top 5 negative

for pc_num in range(1, N_PCS_TO_PLOT + 1):
    plot_single_pc_loadings(
        otu_df_filtered=otu_df_filtered,
        pca_otu_loadings_explore=pca_otu_loadings_explore,
        otu_explained_variance_explore=otu_explained_variance_explore,
        pc_num=pc_num,
        top_k=TOP_K_LOADINGS_EACH,
        save_dir=LOADINGS_OUT_DIR,
        figsize=(9, 7),
    )

Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/all_pc_loadings/pc1_loadings_top10_pos_neg.png
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/all_pc_loadings/pc1_loadings_top10_pos_neg.pdf
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/all_pc_loadings/pc2_loadings_top10_pos_neg.png
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/all_pc_loadings/pc2_loadings_top10_pos_neg.pdf
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/all_pc_loadings/pc3_loadings_top10_pos_neg.png
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/all_pc_loadings/pc3_loadings_top10_pos_neg.pdf
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/all_pc_loadings/pc4_loadings_top10_pos_neg.png
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/all_pc_loadings/pc4_loadings_top10_pos_neg.pdf
Saved: hongrui_result/RF_on_PCA_Results/poster_large_plots/all_pc_loadings/pc5_loadings_top10_pos_neg.png
Saved: hongrui_result/RF_on_PCA_Results/poster